Below is a **complete end-to-end Word2Vec training project** using the **Harry Potter books dataset**. This is a production-style notebook that covers:

* ✅ Loading the dataset
* ✅ Text preprocessing
* ✅ Tokenization
* ✅ Training Word2Vec
* ✅ Saving & Loading the model
* ✅ Finding similar words
* ✅ Word similarity
* ✅ Word analogy
* ✅ Visualizing embeddings with PCA & t-SNE

---

# Step 1: Install Libraries

```python
!pip install gensim nltk scikit-learn matplotlib pandas
```

---

# Step 2: Import Libraries

```python
import re
import nltk
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from nltk.tokenize import sent_tokenize
from gensim.models import Word2Vec
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE

nltk.download("punkt")
```

---

# Step 3: Load Harry Potter Dataset

Assume your folder structure is

```
HarryPotterBooks/

    book1.txt
    book2.txt
    book3.txt
    book4.txt
    book5.txt
    book6.txt
    book7.txt
```

```python
import os

folder = "HarryPotterBooks"

corpus = ""

for file in os.listdir(folder):
    if file.endswith(".txt"):
        with open(os.path.join(folder,file),
                  encoding="utf8",
                  errors="ignore") as f:
            corpus += f.read()
```

---

# Step 4: View Dataset

```python
print(corpus[:1000])
```

---

# Step 5: Basic Cleaning

```python
corpus = corpus.lower()

corpus = re.sub(r'[^a-zA-Z\s]', ' ', corpus)

corpus = re.sub(r'\s+', ' ', corpus)
```

---

# Step 6: Sentence Tokenization

```python
sentences = sent_tokenize(corpus)

print(len(sentences))
```

---

# Step 7: Word Tokenization

```python
data = []

for sentence in sentences:
    words = sentence.split()
    data.append(words)

print(data[:5])
```

Output

```python
[
 ['mr', 'and', 'mrs', 'dursley'],
 ['they', 'were', 'proud'],
 ...
]
```

---

# Step 8: Train Word2Vec Model

```python
model = Word2Vec(

    sentences=data,

    vector_size=100,

    window=5,

    min_count=2,

    workers=4,

    sg=1
)
```

Explanation

```
vector_size = 100

100 dimensional embeddings

window = 5

5 words on left
5 words on right

min_count = 2

Ignore words occurring once

workers = 4

Use four CPU cores

sg = 1

SkipGram

sg = 0

CBOW
```

---

# Step 9: Save Model

```python
model.save("HarryPotterWord2Vec.model")
```

---

# Step 10: Load Model

```python
model = Word2Vec.load("HarryPotterWord2Vec.model")
```

---

# Step 11: Vocabulary Size

```python
len(model.wv.index_to_key)
```

---

# Step 12: View Vocabulary

```python
model.wv.index_to_key[:50]
```

---

# Step 13: Vector of Word

```python
model.wv["harry"]
```

Output

```
100 dimensional vector
```

---

# Step 14: Shape of Vector

```python
model.wv["harry"].shape
```

Output

```python
(100,)
```

---

# Step 15: Most Similar Words

```python
model.wv.most_similar("harry")
```

Output

```
hermione

ron

hagrid

dumbledore

snape
```

---

# Step 16: Similarity Between Two Words

```python
model.wv.similarity("harry","hermione")
```

Output

```
0.84
```

---

# Step 17: Dissimilar Words

```python
model.wv.similarity("harry","voldemort")
```

---

# Step 18: Word Analogy

```python
model.wv.most_similar(

    positive=["king","woman"],

    negative=["man"]

)
```

Output

```
queen
```

Harry Potter Example

```python
model.wv.most_similar(

    positive=["harry","female"],

    negative=["male"]

)
```

---

# Step 19: Doesn't Match

```python
model.wv.doesnt_match(

    ["harry",

     "ron",

     "hermione",

     "car"]

)
```

Output

```
car
```

---

# Step 20: PCA Visualization

```python
words = model.wv.index_to_key[:100]

vectors = np.array([model.wv[word] for word in words])

pca = PCA(n_components=2)

result = pca.fit_transform(vectors)
```

---

# Plot PCA

```python
plt.figure(figsize=(12,10))

for i,word in enumerate(words):

    plt.scatter(result[i,0],result[i,1])

    plt.text(result[i,0],
             result[i,1],
             word)

plt.show()
```

---

# Step 21: t-SNE Visualization

```python
words = model.wv.index_to_key[:200]

vectors = np.array([model.wv[word] for word in words])

tsne = TSNE(

    n_components=2,

    random_state=42,

    perplexity=30

)

result = tsne.fit_transform(vectors)
```

---

# Plot

```python
plt.figure(figsize=(15,12))

for i,word in enumerate(words):

    plt.scatter(result[i,0],result[i,1])

    plt.text(result[i,0],
             result[i,1],
             word)

plt.show()
```

---

# Step 22: Check if Word Exists

```python
"harry" in model.wv
```

Output

```
True
```

---

# Step 23: Vocabulary Frequency

```python
model.wv.get_vecattr("harry","count")
```

---

# Step 24: Training with CBOW

```python
model_cbow = Word2Vec(

    sentences=data,

    vector_size=100,

    window=5,

    min_count=2,

    workers=4,

    sg=0
)
```

---

# Step 25: Compare CBOW vs SkipGram

```python
print(

model.wv.most_similar("harry")

)

print(

model_cbow.wv.most_similar("harry")

)
```

---

# Complete Workflow

```
Harry Potter Books

        │

        ▼

Load Dataset

        │

        ▼

Cleaning

        │

        ▼

Sentence Tokenization

        │

        ▼

Word Tokenization

        │

        ▼

Vocabulary Creation

        │

        ▼

Train Word2Vec

        │

        ▼

Save Model

        │

        ▼

Word Embeddings

        │

        ▼

Similarity Search

        │

        ▼

Visualization
```

# Important Notes

There is one issue in the code above:

```python
corpus = re.sub(r'[^a-zA-Z\s]', ' ', corpus)
sentences = sent_tokenize(corpus)
```

This removes sentence-ending punctuation (`.`, `!`, `?`) **before** sentence tokenization, making `sent_tokenize()` much less effective because it relies on punctuation to detect sentence boundaries.

A better workflow is:

```python
raw_sentences = sent_tokenize(corpus)  # tokenize first

data = []
for sentence in raw_sentences:
    sentence = sentence.lower()
    sentence = re.sub(r'[^a-zA-Z\s]', ' ', sentence)
    sentence = re.sub(r'\s+', ' ', sentence).strip()
    if sentence:
        data.append(sentence.split())
```

This preserves sentence boundaries and produces better training data for Word2Vec. It's the approach I'd recommend for real projects.


---
# `Training Word2Vec Model on Personal Data`
---

Pretrained model --> google --> 3o lakh words --> embeddings --> google news data

Own Dataset --> train our own word2vec model